In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 1


## Varying fad proportion (paramter q)

### Parameters

In [5]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportions = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 15
psi = 15
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [6]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)  
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [7]:
results_dict = {}
for fads_proportion in fads_proportions:
    vec_env = get_as_env(num_trajectories=1000, fads_proportion=fads_proportion)

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(
      env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[fads_proportion] = dict(results=results, rewards=total_rewards, obs=observations)

✓ Pre-computed ODE solutions for 100 time points
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sq

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [8]:
header = f"{'Fads Prop':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for fads_prop, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

 Fads Prop |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       0.0 |      21.15 |       4.87 |           -0.08 | 2.226566864030811
       0.2 |      20.74 |       5.75 |          -0.062 | 2.2570237039074272
       0.4 |      19.49 |       8.14 |          -0.047 | 2.286217618688125
       0.6 |      17.23 |      11.20 |          -0.075 | 2.37305183255655
       0.8 |      14.12 |      15.14 |          -0.063 | 2.474071745119773
         1 |      10.37 |      20.34 |          -0.057 | 2.64419193705752


## Varying eta

### Parameters

In [9]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
etas = [2.5, 5, 7.5, 10.0, 12.5]
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 15
psi = 15
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [10]:
def get_as_env(num_trajectories:int = 1, eta:float=10):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [11]:
results_dict = {}
for eta in etas:
    vec_env = get_as_env(num_trajectories=1000, eta=eta)

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)

✓ Pre-computed ODE solutions for 100 time points
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sq

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad

In [12]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      -5.37 |      44.96 |          -0.113 | 3.2047201125839364
         5 |       9.77 |      21.69 |          -0.083 | 2.6585919205474164
       7.5 |      15.02 |      14.27 |          -0.072 | 2.4463883583764865
      10.0 |      17.23 |      11.20 |          -0.075 | 2.37305183255655
      12.5 |      18.51 |       9.44 |          -0.067 | 2.3290579640704525


## Varying gamma parameter

### Parameters

In [13]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportions = 0.6# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phi = 15
psi = 15
k = 1
gammas = [0, 1, 2, 3]
alpha=0.001
big_phi=0.1
mu=0

In [14]:
def get_as_env(num_trajectories:int = 1, gamma:float=1):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [15]:
results_dict = {}
for gamma in gammas:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma)

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)

✓ Pre-computed ODE solutions for 100 time points
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sq

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")
/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [16]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |      18.97 |      10.85 |          -0.053 | 2.2781990694405962
         1 |      17.23 |      11.20 |          -0.075 | 2.37305183255655
         2 |      15.65 |      12.15 |          -0.073 | 2.526988523915374
         3 |      14.13 |      13.59 |          -0.078 | 2.7323828428681076


## Varying Informed trader proportion (psi and phi)

### Parameters

In [17]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 100
initial_inventory = 0
fill_exponent = 0
fads_proportion = 0.6
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
baseline_arrival_rate = np.array([[5.0, 5.0]])
phis = [30, 22.5, 15, 7.5, 0]
psis = [0, 7.5, 15, 22.5, 30]
k = 1
gamma = 1
alpha=0.001
big_phi=0.1
mu=0

In [18]:
def get_as_env(num_trajectories:int = 1, phi:float=15, psi:float=15):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price,drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories)
    arrival_model = FadsInformedUniformedTradersArrivalModel(baseline_arrival_rate=baseline_arrival_rate,
                                                step_size=terminal_time/n_steps,
                                                phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma, terminal_time=terminal_time,
                                                num_trajectories=num_trajectories)
    fill_probability_model = ExponentialFillFunction(fill_exponent=fill_exponent, 
                                                     step_size=1/n_steps,
                                                     num_trajectories=num_trajectories)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [19]:
print(zip(phis, psis))

In [20]:
results_dict = {}
for phi, psi in zip(phis, psis):
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(
        env=vec_env
    )

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[(phi, psi)] = dict(results=results, rewards=total_rewards,  obs=observations)

✓ Pre-computed ODE solutions for 100 time points
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sqrt(dt) in fad update
case without sq

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:533: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [21]:
header = f"{'Phi':>8} | {'Psi':>8} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for (phi, psi), result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{phi:8} | {psi:8} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Phi |      Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-------------------------------------------------------------------------------
      30 |        0 |      18.97 |      10.85 |          -0.053 | 2.2781990694405962
    22.5 |      7.5 |      18.15 |      10.91 |          -0.064 | 2.3185995773311094
      15 |       15 |      17.23 |      11.20 |          -0.075 | 2.37305183255655
     7.5 |     22.5 |      16.52 |      11.42 |          -0.081 | 2.427022661616492
       0 |       30 |      15.53 |      11.96 |          -0.069 | 2.5203648545399138
